In [24]:
import torch
import torch.nn as nn


class DeformConv2d(nn.Module):
    def __init__(self, inc, outc, kernel_size=3, padding=1, stride=1, bias=None, modulation=False):

        super(DeformConv2d, self).__init__()
        self.kernel_size = kernel_size
        self.padding = padding
        self.stride = stride
        self.zero_padding = nn.ZeroPad2d(padding)
        self.conv = nn.Conv2d(inc, outc, kernel_size=kernel_size, stride=kernel_size, bias=bias)

        self.p_conv = nn.Conv2d(inc, 2 * kernel_size * kernel_size, kernel_size=3, padding=1, stride=stride)
        nn.init.constant_(self.p_conv.weight, 0)
        self.p_conv.register_backward_hook(self._set_lr)

        self.modulation = modulation


    @staticmethod
    def _set_lr(module, grad_input, grad_output):
        grad_input = (grad_input[i] * 0.1 for i in range(len(grad_input)))
        grad_output = (grad_output[i] * 0.1 for i in range(len(grad_output)))

    # 生成卷积中心点 周围的相对坐标: pn
    def _get_p_n(self, N, dtype):
        p_n_x, p_n_y = torch.meshgrid(
            torch.arange(-(self.kernel_size - 1) // 2, (self.kernel_size - 1) // 2 + 1),
            torch.arange(-(self.kernel_size - 1) // 2, (self.kernel_size - 1) // 2 + 1))
        p_n = torch.cat([torch.flatten(p_n_x), torch.flatten(p_n_y)], 0)
        p_n = p_n.view(1, 2 * N, 1, 1).type(dtype)
        return p_n

    # 卷积中心坐标: p0
    def _get_p_0(self, h, w, N, dtype):
        p_0_x, p_0_y = torch.meshgrid(
            torch.arange(1, h * self.stride + 1, self.stride),
            torch.arange(1, w * self.stride + 1, self.stride))
        p_0_x = p_0_x.view(1, 1, h, w).repeat(1, N, 1, 1)
        p_0_y = p_0_y.view(1, 1, h, w).repeat(1, N, 1, 1)
        p_0 = torch.cat([p_0_x, p_0_y], 1).type(dtype)
        return p_0

    # 计算 p0 + pn + offset
    def _get_p(self, offset, dtype):
        # N: 卷积核像素数  h：特征图高  w:特征图宽
        N, h, w = offset.shape[1] // 2, offset.size(2), offset.size(3)
        p_n = self._get_p_n(N, dtype)       # shape = (1, 2 * N, 1, 1)
        p_0 = self._get_p_0(h, w, N, dtype) # 
        p = p_0 + p_n + offset
        return p

    def _get_x_q(self, x, q, N):
        b, h, w, _ = q.size()
        padded_w = x.shape[3]
        c = x.shape[1]
        x = x.contiguous().view(b, c, -1)
        index = q[..., :N] * padded_w + q[..., N:]
        index = index.contiguous().unsqueeze(dim=1).expand(-1, c, -1, -1, -1).contiguous().view(b, c, -1)
        x_offset = x.gather(dim=-1, index=index).contiguous().view(b, c, h, w, N)
        return x_offset

    @staticmethod
    def _reshape_x_offset(x_offset, ks):
        b, c, h, w, N = x_offset.size()
        x_offset = torch.cat([x_offset[..., s:s + ks].contiguous().view(b, c, h, w * ks) for s in range(0, N, ks)], dim=-1)
        x_offset = x_offset.contiguous().view(b, c, h * ks, w * ks)
        return x_offset

    def forward(self, x):          # torch.Size([4, 3, 32, 32])
        offset = self.p_conv(x)    # torch.Size([4, 2*kernel_size^2=18, 32, 32])
        dtype = offset.data.type() # 'torch.FloatTensor'
        ks = self.kernel_size      # 3
        N = offset.shape[1] // 2   # 18//2=9  
        if self.padding: # True
            x = self.zero_padding(x)   # torch.Size([4, 3, 34, 34])
        p = self._get_p(offset, dtype) # torch.Size([4, 18, 32, 32])
        p = p.contiguous().permute(0, 2, 3, 1) # torch.Size([4, 32, 32, 18])
        q_lt = p.detach().floor()
        q_rb = q_lt + 1

        # Clamps all elements in input into the range [ min, max ]
        q_lt = torch.cat([torch.clamp(q_lt[..., :N], 0, x.size(2) - 1), torch.clamp(q_lt[..., N:], 0, x.size(3) - 1)], dim=-1).long()
        q_rb = torch.cat([torch.clamp(q_rb[..., :N], 0, x.size(2) - 1), torch.clamp(q_rb[..., N:], 0, x.size(3) - 1)], dim=-1).long()
        q_lb = torch.cat([q_lt[..., :N], q_rb[..., N:]], dim=-1)
        q_rt = torch.cat([q_rb[..., :N], q_lt[..., N:]], dim=-1)

        # clip p
        p = torch.cat([torch.clamp(p[..., :N], 0, x.size(2) - 1), torch.clamp(p[..., N:], 0, x.size(3) - 1)], dim=-1)


        g_lt = (1 + (q_lt[..., :N].type_as(p) - p[..., :N])) * (1 + (q_lt[..., N:].type_as(p) - p[..., N:]))
        g_rb = (1 - (q_rb[..., :N].type_as(p) - p[..., :N])) * (1 - (q_rb[..., N:].type_as(p) - p[..., N:]))
        g_lb = (1 + (q_lb[..., :N].type_as(p) - p[..., :N])) * (1 - (q_lb[..., N:].type_as(p) - p[..., N:]))
        g_rt = (1 - (q_rt[..., :N].type_as(p) - p[..., :N])) * (1 + (q_rt[..., N:].type_as(p) - p[..., N:]))

        # 获取 偏移像素点最邻近的4个像素点的坐标
        x_q_lt = self._get_x_q(x, q_lt, N)
        x_q_rb = self._get_x_q(x, q_rb, N)
        x_q_lb = self._get_x_q(x, q_lb, N)
        x_q_rt = self._get_x_q(x, q_rt, N)

        # 计算双线性差值后的像素值
        x_offset = g_lt.unsqueeze(dim=1) * x_q_lt + \
                   g_rb.unsqueeze(dim=1) * x_q_rb + \
                   g_lb.unsqueeze(dim=1) * x_q_lb + \
                   g_rt.unsqueeze(dim=1) * x_q_rt


        x_offset = self._reshape_x_offset(x_offset, ks) # torch.Size([4, 3, 96, 96])
        out = self.conv(x_offset)
        return out # torch.Size([4, 5, 32, 32])
#################----------------------##################
import time
x = torch.randn((4, 3, 32, 32))
deformable_conv = DeformConv2d(inc=3, outc=5, kernel_size=3)
time_begin = time.time()
result = deformable_conv(x)
print(f'cast_time:',time.time()-time_begin)
print('result.shape:',result.shape)


f:\my_softers\project_IDE\Anconda\envs\jp_layout_pytorch\lib\site-packages\torch\nn\modules\module.py:795: UserWarning: Using a non-full backward hook when the forward contains multiple autograd Nodes is deprecated and will be removed in future versions. This hook will be missing some grad_input. Please use register_full_backward_hook to get the documented behavior.
  warnings.warn("Using a non-full backward hook when the forward contains multiple autograd Nodes "


cast_time: 143.3515076637268
result.shape: torch.Size([4, 5, 32, 32])


In [19]:
import time
x = torch.randn((4, 3, 32, 32))
deformable_conv = DeformConv2d(inc=3, outc=5, kernel_size=3)
time_begin = time.time()
result = deformable_conv(x)
print(f'cast_time:',time.time()-time_begin)
print('result.shape:',result.shape)


cast_time: 0.023936033248901367
result.shape: torch.Size([4, 5, 32, 32])


f:\my_softers\project_IDE\Anconda\envs\jp_layout_pytorch\lib\site-packages\torch\nn\modules\module.py:795: UserWarning: Using a non-full backward hook when the forward contains multiple autograd Nodes is deprecated and will be removed in future versions. This hook will be missing some grad_input. Please use register_full_backward_hook to get the documented behavior.
  warnings.warn("Using a non-full backward hook when the forward contains multiple autograd Nodes "


In [20]:
conv2d = nn.Conv2d(3,5,3)
time_begin = time.time()
result = conv2d(x)
print(f'cast_time:',time.time()-time_begin)
print('result.shape:',result.shape)

cast_time: 0.0009968280792236328
result.shape: torch.Size([4, 5, 30, 30])
